# Phase 1 — Mediation (Round 1 + Optional Critique Round)

**Flow:**
1. Load opinions CSV → run round 1 → see winning statement
2. **STOP** — collect critiques into a spreadsheet
3. Load critiques CSV → run round 2 → get final statement + voting form template

Round 2 is optional. If the round 1 statement looks good, skip straight to `02_analyse.ipynb`.

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
TOPIC        = "Where should we hold the combined summer social for engineering and marketing?"
OPINIONS_CSV = "example_opinions.csv"   # ← your opinions form export

NUM_CANDIDATES = 6   # statements generated per round
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import sys, os, json, asyncio
from pathlib import Path
from uuid import UUID

import nest_asyncio
nest_asyncio.apply()

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

import pandas as pd
from group_consensus.models.types import (
    Opinion, Participant, SessionConfig, Statement, StatementType
)
from group_consensus.mediation.async_mediator import (
    AsyncStatementModel, AsyncRewardModel
)
from group_consensus.mediation.social_choice import select_winner_from_rankings

OUTPUT_DIR = Path("session_data")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"API key : {'✓' if os.getenv('ANTHROPIC_API_KEY') else '⚠  not found'}")

---
## PHASE 1A — Load opinions and run round 1

In [ ]:
df = pd.read_csv(OPINIONS_CSV)
print(f"{len(df)} responses — columns: {list(df.columns)}")
df

In [ ]:
# Adjust column indices if your form uses different column names
NAME_COL    = df.columns[1]
OPINION_COL = df.columns[2]

print(f"Name column    : '{NAME_COL}'")
print(f"Opinion column : '{OPINION_COL}'")

In [ ]:
SESSION_ID   = "session_01"
participants = []
opinions     = []

for i, row in df.iterrows():
    pid  = f"p{i}"
    name = str(row[NAME_COL]).strip()
    text = str(row[OPINION_COL]).strip()
    participants.append(Participant(id=pid, name=name))
    opinions.append(Opinion(participant_id=pid, text=text, session_id=SESSION_ID))

# Name → participant_id lookup (used later for matching critique CSV)
name_to_pid = {p.name: p.id for p in participants}

config = SessionConfig(
    session_id=SESSION_ID,
    topic=TOPIC,
    num_candidate_statements=NUM_CANDIDATES,
    max_deliberation_rounds=1,   # one round at a time — we pause for critiques
)
stmt_model   = AsyncStatementModel(config)
reward_model = AsyncRewardModel(config)

print(f"✓ {len(participants)} participants ready")

In [ ]:
print("Generating candidates...")
r1_candidates = asyncio.run(
    stmt_model.generate_candidates(
        topic=TOPIC, opinions=opinions, participants=participants, round_number=0
    )
)

print(f"Ranking {len(r1_candidates)} candidates for {len(participants)} participants (concurrent)...")
r1_rankings = asyncio.run(
    reward_model.predict_rankings(
        participants=participants,
        opinions=opinions,
        candidates=r1_candidates,
        session_id=SESSION_ID,
        round_number=0,
    )
)

r1_winner_id = select_winner_from_rankings(
    statement_ids=[s.id for s in r1_candidates],
    participant_rankings=[r.statement_ids for r in r1_rankings],
)
r1_winner = next(s for s in r1_candidates if s.id == r1_winner_id)
print("Done.")

In [ ]:
# Show all candidates so you can see what the model generated
print(f"{'═'*60}")
print(f"  Round 1 — all {len(r1_candidates)} candidates")
print(f"{'═'*60}")
for i, s in enumerate(r1_candidates, 1):
    marker = "  ← Schulze winner" if s.id == r1_winner.id else ""
    print(f"\n  {i}. {s.text}{marker}")

print(f"\n{'═'*60}")
print("  ROUND 1 WINNER")
print(f"{'═'*60}")
print(f"\n  \"{r1_winner.text}\"")

In [ ]:
# Save state so you can restart the kernel and still run phase 1B
state = {
    "session_id"  : SESSION_ID,
    "topic"       : TOPIC,
    "participants": [{"id": p.id, "name": p.name} for p in participants],
    "opinions"    : [{"participant_id": o.participant_id, "text": o.text} for o in opinions],
    "r1_winner"   : {"id": str(r1_winner.id), "text": r1_winner.text},
    "r1_candidates": [{"id": str(s.id), "text": s.text} for s in r1_candidates],
}
with open(OUTPUT_DIR / "round_1_state.json", "w") as f:
    json.dump(state, f, indent=2)
print("✓ State saved to session_data/round_1_state.json")

---
## ⏸  STOP HERE — collect critiques

Share the round 1 winner with participants and ask for critiques.
The question to ask is:

> *"What's missing or wrong with this statement? What would you change?  
> (If you're happy with it, leave the critique column blank.)"*

Collect responses into a spreadsheet with two columns:

| Name | Critique |
|------|----------|
| James Okafor | I like the early finish mention but nothing about actually mixing the teams. |
| Sarah Chen | (leave blank if happy) |

Save it as `critiques.csv` in this `notebooks/` folder, then run the cells below.

**If round 1 looks good enough, skip to `02_analyse.ipynb` — a second round is optional.**

---

## PHASE 1B — Load critiques and run round 2

You can run this in the same kernel session (state is in memory) or after a kernel
restart (state is reloaded from `session_data/round_1_state.json`).

In [ ]:
CRITIQUES_CSV = "critiques.csv"   # ← your critique spreadsheet export

In [ ]:
# Reload state if kernel was restarted
try:
    r1_winner
    print("Using in-memory state from phase 1A.")
except NameError:
    print("Reloading state from session_data/round_1_state.json...")
    with open(OUTPUT_DIR / "round_1_state.json") as f:
        state = json.load(f)

    SESSION_ID   = state["session_id"]
    TOPIC        = state["topic"]
    participants = [Participant(**p) for p in state["participants"]]
    opinions     = [
        Opinion(participant_id=o["participant_id"], text=o["text"], session_id=SESSION_ID)
        for o in state["opinions"]
    ]
    name_to_pid  = {p.name: p.id for p in participants}
    r1_winner    = Statement(
        id=UUID(state["r1_winner"]["id"]),
        text=state["r1_winner"]["text"],
        type=StatementType.REFINED,
        session_id=SESSION_ID,
        round_number=0,
    )
    config = SessionConfig(
        session_id=SESSION_ID,
        topic=TOPIC,
        num_candidate_statements=NUM_CANDIDATES,
        max_deliberation_rounds=1,
    )
    stmt_model   = AsyncStatementModel(config)
    reward_model = AsyncRewardModel(config)
    print("✓ State reloaded.")

In [ ]:
crit_df = pd.read_csv(CRITIQUES_CSV)
print(f"{len(crit_df)} rows loaded — columns: {list(crit_df.columns)}")
crit_df

In [ ]:
# Adjust if your column names differ
CRIT_NAME_COL    = crit_df.columns[0]   # Name column
CRIT_TEXT_COL    = crit_df.columns[1]   # Critique column

critiques = []
skipped   = []

for _, row in crit_df.iterrows():
    name = str(row[CRIT_NAME_COL]).strip()
    text = str(row[CRIT_TEXT_COL]).strip() if pd.notna(row[CRIT_TEXT_COL]) else ""

    if not text or text.lower() in ("", "nan", "none", "-", "n/a", "happy", "ok", "fine"):
        skipped.append(name)
        continue

    pid = name_to_pid.get(name)
    if pid is None:
        print(f"  ⚠  '{name}' not found in participant list — check spelling")
        continue

    critiques.append(Opinion(participant_id=pid, text=text, session_id=SESSION_ID))

print(f"\n{len(critiques)} critique(s) loaded:")
for c in critiques:
    name = next(p.name for p in participants if p.id == c.participant_id)
    print(f"  {name}: {c.text}")

if skipped:
    print(f"\n{len(skipped)} participant(s) satisfied (blank critique): {', '.join(skipped)}")

if not critiques:
    print("\n⚠  No critiques found — everyone appears satisfied. "
          "Skip to 02_analyse.ipynb and use the round 1 winner.")

In [ ]:
print("Generating round 2 candidates (informed by critiques)...")
r2_candidates = asyncio.run(
    stmt_model.generate_candidates(
        topic=TOPIC,
        opinions=opinions,        # original opinions still in play
        participants=participants,
        critiques=critiques,      # targeted critiques of round 1 winner
        previous_winner=r1_winner,
        round_number=1,
    )
)

print(f"Ranking for {len(participants)} participants...")
r2_rankings = asyncio.run(
    reward_model.predict_rankings(
        participants=participants,
        opinions=opinions,        # rank against original opinions, not critiques
        candidates=r2_candidates,
        session_id=SESSION_ID,
        round_number=1,
    )
)

r2_winner_id = select_winner_from_rankings(
    statement_ids=[s.id for s in r2_candidates],
    participant_rankings=[r.statement_ids for r in r2_rankings],
)
r2_winner = next(s for s in r2_candidates if s.id == r2_winner_id)
r2_winner.type = StatementType.REFINED
print("Done.")

In [ ]:
print(f"{'═'*60}")
print(f"  Round 2 — all {len(r2_candidates)} candidates")
print(f"{'═'*60}")
for i, s in enumerate(r2_candidates, 1):
    marker = "  ← Schulze winner" if s.id == r2_winner.id else ""
    print(f"\n  {i}. {s.text}{marker}")

print(f"\n{'═'*60}")
print("  BEFORE (round 1)")
print(f"{'═'*60}")
print(f"  \"{r1_winner.text}\"")
print(f"\n{'═'*60}")
print("  AFTER (round 2 — critique-informed)")
print(f"{'═'*60}")
print(f"  \"{r2_winner.text}\"")

In [ ]:
# Save all statements for 02_analyse.ipynb
final_winner   = r2_winner
voting_stmts   = r2_candidates   # final round candidates go on the voting form

all_stmts = r1_candidates + [s for s in r2_candidates if s.id not in {x.id for x in r1_candidates}]

session_data = {
    "session_id"            : SESSION_ID,
    "topic"                 : TOPIC,
    "consensus_statement_id": str(final_winner.id),
    "statements"            : [
        {"id": str(s.id), "text": s.text, "type": s.type, "round": s.round_number}
        for s in all_stmts
    ],
}
with open(OUTPUT_DIR / "statements.json", "w") as f:
    json.dump(session_data, f, indent=2)

label_map = {
    f"Statement {i}": {"id": str(s.id), "text": s.text}
    for i, s in enumerate(voting_stmts, 1)
}
with open(OUTPUT_DIR / "label_map.json", "w") as f:
    json.dump(label_map, f, indent=2)

print("✓ Saved statements.json and label_map.json")
print()

# Print voting form template
DIVIDER = "═" * 60
print(DIVIDER)
print("  VOTING FORM TEMPLATE")
print(DIVIDER)
print(f"\nForm title: {TOPIC}")
print("Add a short-answer question: 'Your name'")
print("Then one multiple-choice question per statement (Agree / Pass / Disagree):\n")
for label, info in label_map.items():
    print(f"  {label}")
    print(f"  {info['text']}")
    print()